# Ingestion Web

### WebBaseLoader

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

In [ ]:
loader = WebBaseLoader("https://okandata.com/")

docs = loader.load()
docs[0].page_content[:3000]

### Nettoyage

In [ ]:
import re

def clean_web_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)               # espaces multiples
    text = re.sub(r"\n+", "\n", text)              # sauts de ligne multiples
    text = re.sub(r"(Cookies?|Privacy Policy).*?$", "", text, flags=re.I)
    text = text.strip()
    return text


# Split Web

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

cleaned_docs = []

for d in docs:
    d.page_content = clean_web_text(d.page_content)
    cleaned_docs.append(d)

web_chunks = splitter.split_documents(cleaned_docs)

len(web_chunks)


In [ ]:
web_chunks[0].page_content

## Créer une classe

In [ ]:
class ingestWeb:
    def __init__(self, chunk_size = 800, chunk_overlap= 100):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=800,
            chunk_overlap=100,
            separators=["\n\n", "\n", ".", " ", ""]
)
    def _clean(self, text :str) ->str:
        return clean_web_text(text)
    
    def process(self, url : str):
        docs = WebBaseLoader(url).load()
        for d in docs:
            d.page_content = self._clean(d.page_content)

        return self.splitter.split_documents(docs)

# Application

In [ ]:
ingest_web = ingestWeb()
chunks = ingest_web.process("https://okandata.com/")

print("Chunks générés :", len(chunks))
chunks[0].page_content